# Magnetic field evolution fit

This notebook reproduces Figure 2 in [Graber et al. (2024)](https://arxiv.org/abs/2312.14848).

For details see also Appendix A in the paper.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.magneto_rotational_physics.magneto_rotational_evolution_fit as mre
import utilities.plot_settings

from matplotlib import rc

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 60

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=MEDIUM_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In the following, we fit analytical functions to simulated magnetic-field evolution curves. The functional form we use is the following: 

if $\tau_1 < \tau_2 < \tau_{\rm late}$:

$$B(t) = B_{\rm initial} (1 + t/\tau_1)^{a_1} (1 + t/\tau_2)^{a_2-a_1} (1 + t/\tau_{\rm late})^{a_{\rm late}-a_2}$$ 

if $\tau_1 < \tau_{\rm late} < \tau_2$:

$$B(t) = B_{\rm initial} (1 + t/\tau_1)^{a_1} (1 + t/\tau_{\rm late})^{a_{\rm late}-a_1}$$ 

if $\tau_{\rm late} < \tau_1 < \tau_2$:

$$B(t) = B_{\rm initial} (1 + t/\tau_{\rm late})^{a_{\rm late}}$$ 

where $\tau_1 = A_1  B_{\rm initial}^{b_1}$ and $\tau_2 = A_2 B_{\rm initial}^{b_2}$ and $\tau_{\rm late}$ is a constant.

## Load the simulated magnetic field evolution curves

For the magneto-thermal simulations the following set-up was employed: The equation of state is SLy4 with a NS mass of 1.4 Msun and radius of 11.74 km. The impurity parameter in the pasta layer is fixed to 100. For the impurity in the outer and inner crust(excluding the pasta layer), the fits of [Carreau et al.(2020)](https://ui.adsabs.harvard.edu/abs/2020A%26A...640A..77C/abstract) have been used (see Figure 5 in that paper). The envelope model is taken from [Potekhin et al. (2015)](https://ui.adsabs.harvard.edu/abs/2015SSRv..191..239P/abstract). Superfluid and superconducting gap parametrisations are taken from [Ho et al. (2015)](https://ui.adsabs.harvard.edu/abs/2015SciA....1E0578H/abstract): SFB for crustal neutrons, TToa for core neutrons and CCDKp for core protons. 

The initial magnetic field of the simulated curves are: $10^{12}$ G, $10^{13}$ G, $10^{14}$ G, $10^{15}$ G, $3 \times 10^{15}$ G.

In [ ]:
df_simB12 = pd.read_csv(
    "../../mlpoppyns/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e12_Btor1e13.csv",
    delimiter=",",
    header=[0],
)
df_simB12.head()

In [ ]:
df_simB13 = pd.read_csv(
    "../../mlpoppyns/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e13_Btor1e14.csv",
    delimiter=",",
    header=[0],
)
df_simB13.head()

In [ ]:
df_simB14 = pd.read_csv(
    "../../mlpoppyns/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e14_Btor1e15.csv",
    delimiter=",",
    header=[0],
)
df_simB14.head()

In [ ]:
df_simB15 = pd.read_csv(
    "../../mlpoppyns/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip1e15_Btor1e16.csv",
    delimiter=",",
    header=[0],
)
df_simB15.head()

In [ ]:
df_simB5e15 = pd.read_csv(
    "../../mlpoppyns/simulator/magneto_rotational_physics/magneto-thermal_evol_curves/cool_curve_CC_Bdip5e15_Btor1e16.csv",
    delimiter=",",
    header=[0],
)
df_simB5e15.head()

In [ ]:
t12_sim = df_simB12["t[yr]"].to_numpy().astype(np.float64)
t13_sim = df_simB13["t[yr]"].to_numpy().astype(np.float64)
t14_sim = df_simB14["t[yr]"].to_numpy().astype(np.float64)
t15_sim = df_simB15["t[yr]"].to_numpy().astype(np.float64)
t5e15_sim = df_simB5e15["t[yr]"].to_numpy().astype(np.float64)

B12_sim = df_simB12["B[G]"].to_numpy().astype(np.float64)
B13_sim = df_simB13["B[G]"].to_numpy().astype(np.float64)
B14_sim = df_simB14["B[G]"].to_numpy().astype(np.float64)
B15_sim = df_simB15["B[G]"].to_numpy().astype(np.float64)
B5e15_sim = df_simB5e15["B[G]"].to_numpy().astype(np.float64)

## Evolution curves from the analytical fit model

In [ ]:
log_B_initial = np.array([9, 10, 11, 12, 13, 14, 15, np.log10(5.0e15), 16, 17])
B_initial = 10**log_B_initial
time = np.logspace(0.0, 9.0, 100)
a_late = -3.0

B_asymptotic = 10 ** np.random.normal(
    cfg["B_millisec_mean"], cfg["B_millisec_sigma"], len(B_initial)
)

B9_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[0], time, B_asymptotic[0], a_late
)
B10_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[1], time, B_asymptotic[1], a_late
)
B11_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[2], time, B_asymptotic[2], a_late
)
B12_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[3], time, B_asymptotic[3], a_late
)
B13_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[4], time, B_asymptotic[4], a_late
)
B14_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[5], time, B_asymptotic[5], a_late
)
B15_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[6], time, B_asymptotic[6], a_late
)
B5e15_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[7], time, B_asymptotic[7], a_late
)
B16_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[8], time, B_asymptotic[8], a_late
)
B17_fit = mre.magnetic_field_evolution_fit_numpy(
    B_initial[9], time, B_asymptotic[9], a_late
)

In [ ]:
# Plot the magnetic field evolution curves.
fig, ax = plt.subplots(figsize=(12, 10))

colors = plt.get_cmap("viridis", len(log_B_initial))
norm = mpl.colors.Normalize(
    vmin=np.min(log_B_initial), vmax=np.max(log_B_initial) + 1
)
sm = plt.cm.ScalarMappable(norm=norm, cmap=colors)
sm.set_array([])

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0, 1.0e9)
ax.set_xlabel(r"Time $t$ [yr]")
ax.set_ylabel(r"Dipolar magnetic field $B$ [G]")

ax.plot(
    t12_sim,
    B12_sim,
    linestyle="-",
    linewidth=3,
    color=colors(3),
    rasterized=True,
)
ax.plot(
    t13_sim,
    B13_sim,
    linestyle="-",
    linewidth=3,
    color=colors(4),
    rasterized=True,
)
ax.plot(
    t14_sim,
    B14_sim,
    linestyle="-",
    linewidth=3,
    color=colors(5),
    rasterized=True,
)
ax.plot(
    t15_sim,
    B15_sim,
    linestyle="-",
    linewidth=3,
    color=colors(6),
    rasterized=True,
)
ax.plot(
    t5e15_sim,
    B5e15_sim,
    linestyle="-",
    linewidth=3,
    color=colors(7),
    rasterized=True,
)

ax.plot(
    time,
    B9_fit,
    linestyle="--",
    linewidth=4,
    color=colors(0),
    rasterized=True,
)
ax.plot(
    time,
    B10_fit,
    linestyle="--",
    linewidth=4,
    color=colors(1),
    rasterized=True,
)

ax.plot(
    time,
    B11_fit,
    linestyle="--",
    linewidth=4,
    color=colors(2),
    rasterized=True,
)
ax.plot(
    time,
    B12_fit,
    linestyle="--",
    linewidth=4,
    color=colors(3),
    rasterized=True,
)
ax.plot(
    time,
    B13_fit,
    linestyle="--",
    linewidth=4,
    color=colors(4),
    rasterized=True,
)
ax.plot(
    time,
    B14_fit,
    linestyle="--",
    linewidth=4,
    color=colors(5),
    rasterized=True,
)
ax.plot(
    time,
    B15_fit,
    linestyle="--",
    linewidth=4,
    color=colors(6),
    rasterized=True,
)
ax.plot(
    time,
    B5e15_fit,
    linestyle="--",
    linewidth=4,
    color=colors(7),
    rasterized=True,
)
ax.plot(
    time,
    B16_fit,
    linestyle="--",
    linewidth=4,
    color=colors(8),
    rasterized=True,
)
ax.plot(
    time,
    B17_fit,
    linestyle="--",
    linewidth=4,
    color=colors(9),
    rasterized=True,
)

plt.tight_layout()
plt.savefig("../../paper_plots/graber_etal_2024/plots/B_fields.pdf", dpi=400, bbox_inches="tight")
plt.show()